In [ ]:
#| hide
from compact import lisp


In [ ]:
lisp.register_magic()

In [ ]:
def foo(a, b, c, **kw):
    print(locals())

In [ ]:
foo(1, 2, 3, x=5, y=6)

{'a': 1, 'b': 2, 'c': 3, 'kw': {'x': 5, 'y': 6}}


how do we call foo from inside lisp?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me check the module's docs for the calling convention:
- ⏳ `py(code="doc(lisp)")` ⏳


```json {.tool}
{
  "id": "toolu_01FPdkjGRwFTNVpEPL53LZAe",
  "name": "py",
  "args": {
    "code": "doc(lisp)"
  },
  "result": "LispCtx instance: <compact.core.LispCtx object at 0x770f2a48f1a0>\"\"\"convenience lisp context for interop\"\"\""
}
```

The doc for `lisp` itself is sparse — it's just a `LispCtx` instance for interop, not a documented API surface. Let me look at the underlying class/module for more detail:
- ⏳ `py(code="doc(lisp.__class__)")` ⏳


```json {.tool}
{
  "id": "toolu_01FZDCfS8RHoKTfXc2syshGW",
  "name": "py",
  "args": {
    "code": "doc(lisp.__class__)"
  },
  "result": "class LispCtx:\n    \"\"\"convenience lisp context for interop\"\"\"\n    def __init__(self): ...\n    def register_magic(self): ...\n    def sf(self, nm=None): ..."
}
```


- ⏳ `py(code="doc(lisp.sf)")` ⏳

- ⏳ `py(code="import compact.core as core\ndoc(core)")` ⏳


```json {.tool}
{
  "id": "toolu_01YANzmfCavzA6wbEfAooz9t",
  "name": "py",
  "args": {
    "code": "doc(lisp.sf)"
  },
  "result": "def sf(\n    nm:NoneType=None\n):\"\"\"Call self as a function.\"\"\""
}
```



```json {.tool}
{
  "id": "toolu_01K7pUR4VxPY3jhMiipJuKCf",
  "name": "py",
  "args": {
    "code": "import compact.core as core\ndoc(core)"
  },
  "result": "# module compact.core:\n\n\"\"\"Scheme implementation in Python\n\nDocs: https://thechandru.github.io/compact/core.html.md\n\"\"\"\n\n## types:\n- class Env: ...  # implementation of env for scheme eval().\n- class Procedure: ...  # a dual use wrapper for scm_eval as well as python callers\n- class LispCtx: ...  # convenience lisp context for interop\n\n## functions:\n- def scm_apply(fn, args)\n- def scm_eval_one_step(expr, env, sfs=()): ...  # eval-uate lisp expressions given an environment and handlers for special forms\n- def scm_eval_tco(expr, env, sfs=()): ...  # eval but with tail call optimization"
}
```


- ⏳ `py(code="doc(core.Env)")` ⏳

- ⏳ `py(code="doc(core.Procedure)")` ⏳

- ⏳ `py(code="doc(lisp.__class__.__init__)\ninfo_md(lis…")` ⏳


```json {.tool}
{
  "id": "toolu_019Qg1MnT4BkogbHCAneoQRx",
  "name": "py",
  "args": {
    "code": "doc(core.Env)"
  },
  "result": "class Env:\n    \"\"\"implementation of env for scheme eval().\"\"\"\n    def __init__(self, d, primitives: dict = (), parent=None): ...\n    def find(self, k): ...\n    def is_primitive(self, k): ...\n    def new_frame(self, bindings=()): ...\n    def update(self, d): ..."
}
```



```json {.tool}
{
  "id": "toolu_0191u6mmL9ZpRczBJbXVMGqL",
  "name": "py",
  "args": {
    "code": "doc(core.Procedure)"
  },
  "result": "class Procedure:\n    \"\"\"a dual use wrapper for scm_eval as well as python callers\"\"\"\n    def __init__(self, params, body, env, sfs): ..."
}
```



`````json {.tool}
{
  "id": "toolu_01McPbVqWvYHCwdDhDgZnkCd",
  "name": "py",
  "args": {
    "code": "doc(lisp.__class__.__init__)\ninfo_md(lisp.__class__.__init__, source=True)"
  },
  "result": "````python\ndef __init__(self):\n    self.env = Env(globals(), primitives=_builtin)\n    self.sfs = {}\n````\n\n**File:** `~/compact/compact/core.py`; line: 178"
}
`````


- ⏳ `py(code="info_md(lisp.__class__.register_magic, s…")` ⏳

- ⏳ `py(code="info_md(core.Env.find, source=True)")` ⏳


`````json {.tool}
{
  "id": "toolu_01K9ANG6KBATezYBDsP9JXYv",
  "name": "py",
  "args": {
    "code": "info_md(lisp.__class__.register_magic, source=True)"
  },
  "result": "````python\ndef register_magic(self):        \n    from IPython.core.magic import register_cell_magic\n    caller_globals = inspect.stack()[1][0].f_globals\n    def _magic(line, cell):\n        exprs = parse_all(cell)\n        expr = exprs[0] if len(exprs) == 1 else [Symbol(\"begin\")] + exprs\n        env = Env(caller_globals, primitives=self.env.primitives, parent=self.env)\n        return scm_eval_tco(expr, env, self.sfs)\n    register_cell_magic('lisp')(_magic)\n````\n\n**File:** `~/compact/compact/core.py`; line: 200"
}
`````



`````json {.tool}
{
  "id": "toolu_0151cykdfYWBFxRxuCTPf99d",
  "name": "py",
  "args": {
    "code": "info_md(core.Env.find, source=True)"
  },
  "result": "````python\ndef find(self, k):\n    if k in self.d | self.primitives: return self\n    if self.parent is not None: return self.parent.find(k)\n    return None\n````\n\n**File:** `~/compact/compact/core.py`; line: 37"
}
`````


- ⏳ `py(code="info_md(core.scm_apply, source=True)")` ⏳

- ⏳ `py(code="info_md(core.Procedure, source=True)")` ⏳


`````json {.tool}
{
  "id": "toolu_01W7ixv3ZWPKPspnRxTM7EGy",
  "name": "py",
  "args": {
    "code": "info_md(core.scm_apply, source=True)"
  },
  "result": "````python\ndef scm_apply(fn, args):\n    if isinstance(fn, Procedure): return fn._thunk(*args)\n    return fn(*args)\n````\n\n**File:** `~/compact/compact/core.py`; line: 50"
}
`````



`````json {.tool}
{
  "id": "toolu_015KpNBAem8rb4eCiPqVkKrQ",
  "name": "py",
  "args": {
    "code": "info_md(core.Procedure, source=True)"
  },
  "result": "````python\nclass Procedure:\n    \"a dual use wrapper for scm_eval as well as python callers\"\n    def __init__(self, params, body, env, sfs): store_attr()\n    def _thunk(self, *args): \n        \"return a Thunk so this can be executed inside the eval loop\"\n        return Thunk(_body_expr(self.body), self.env.new_frame(_bind_params(self.params, args)))\n    def __call__(self, *args): \n        \"if called directly from python, eval it for the caller\"\n        return scm_eval_tco(_body_expr(self.body), \n            self.env.new_frame(_bind_params(self.params, args)), self.sfs)\n````\n\n**File:** `~/compact/compact/core.py`; line: 55"
}
`````

Since `foo` is defined in the notebook's global namespace, `register_magic()` captures those globals as the base environment for `%%lisp` cells (via `Env(caller_globals, ...)`), so you can call `foo` directly by name using Scheme syntax:

```
%%lisp
(foo 1 2 3)
```

This calls `foo(1, 2, 3)` via `scm_apply`, which does `fn(*args)` for plain Python functions. Note `kw` won't be reachable this way — `scm_apply` only passes positional args, so there's no lisp-side syntax here for keyword arguments; `foo`'s `**kw` would just receive `{}`.

```json {.usage}
{"model": "claude-sonnet-5", "prompt_tokens": 372963, "completion_tokens": 2712, "total_tokens": 375675, "cached_tokens": 52170, "cache_creation_tokens": 320779, "cost": 0.057978450000000015}
```


do we need a special lisp builtin to pass in kw?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me check `_bind_params` and the builtin primitives to see if kwargs are supported anywhere in the lisp layer:
- ⏳ `py(code="import compact.core as core\ninfo_md(core…")` ⏳

- ⏳ `py(code="info_md(core._builtin, source=True)")` ⏳


`````json {.tool}
{
  "id": "toolu_019tDLwab6BvEYFKLtxqYEco",
  "name": "py",
  "args": {
    "code": "import compact.core as core\ninfo_md(core._bind_params, source=True)"
  },
  "result": "````python\ndef _bind_params(ps, vs):\n    match ps:\n        case Symbol(s=nm): return {nm: list(vs)}\n        case [*fixed, Symbol(s=\".\"), Symbol(s=rest)]:\n            return dict(zip([p.s for p in fixed], vs)) | {rest: list(vs[len(fixed):])}\n        case _:\n            if len(vs) != len(ps): raise ValueError(f\"arity mismatch: wanted {len(ps)}, got {len(vs)}\")\n            return dict(zip([p.s for p in ps], vs))\n````\n\n**File:** `~/compact/compact/core.py`; line: 226"
}
`````



`````json {.tool}
{
  "id": "toolu_011bHmFapZipkgVC6rnNdga9",
  "name": "py",
  "args": {
    "code": "info_md(core._builtin, source=True)"
  },
  "result": "````python\ndict instance: {'+': <function <lambda> at 0x770f20254540>, '*': <function <lambda> at 0x770f2…\n    \"dict() -> new empty dictionary\ndict(mapping) -> new dictionary initialized from a mapping object's\n    (key, value) pairs\ndict(iterable) -> new dictionary initialized as if via:\n    d = {}\n    for k, v in iterable:\n        d[k] = v\ndict(**kwargs) -> new dictionary initialized with the name=value pairs\n    in the keyword argument list.  For example:  dict(one=1, two=2)\"\n````"
}
`````

Let me get the full builtin dict, since `info_md` truncated it:
- ⏳ `py(code="list(core._builtin.keys())")` ⏳


```json {.tool}
{
  "id": "toolu_01GctLHvvVgkLhm9vd32WZ6h",
  "name": "py",
  "args": {
    "code": "list(core._builtin.keys())"
  },
  "result": "['+', '*', '-', '/', '=', '<', '>', '<=', '>=', 'number?', 'string?', 'symbol?', 'boolean?', 'null?', 'pair?', 'list?', 'list', 'cons', 'car', 'cdr', 'string-append', 'string-length', 'substring', 'number->string', 'string->number', 'floor', 'ceiling', 'round', 'truncate', 'abs', 'min', 'max', 'sqrt', 'expt', 'modulo', 'remainder', 'symbol->string', 'string->symbol', 'equal?', 'not', 'error', 'apply', 'map', 'filter', 'for-each', 'fold-left', 'fold-right']"
}
```

No — there's no existing builtin for it, and none is needed as a "special" lisp form. Looking at `_bind_params` (line 226) and the full builtin list, `scm_apply`/`Procedure.__call__` only ever call `fn(*args)` — no builtin does `**kwargs` splatting into Python.

The `_builtin` dict has 39 entries (`+`, `*`, `apply`, `map`, `filter`, etc.) — none pass keyword arguments through.

So there's no way to invoke `foo`'s `**kw` from `%%lisp` right now. To support it you'd need to add a new primitive, e.g. one that takes a Python dict (built from an alist) and does `fn(*args, **kwargs)`, something like:

```python
core._builtin['apply-kw'] = lambda fn, args, kwargs: fn(*args, **kwargs)
```

then represent the keyword pairs as a Scheme alist and convert them to a dict before calling. There's no such conversion helper in the module currently, so you'd write it yourself — nothing here inherently blocks it, it's just unimplemented.

```json {.usage}
{"model": "claude-sonnet-5", "prompt_tokens": 167625, "completion_tokens": 944, "total_tokens": 168569, "cached_tokens": 119307, "cache_creation_tokens": 48312, "cost": 0.02585175}
```


so something like `(P (@ ('href "/foo"), ('cls "p-5")))`

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a clean idea — using `@` as a marker for a kwargs alist, similar to Hiccup-style syntax. It would need two pieces:

1. A sentinel type, say `KwArgs(dict)`, that a builtin `@` constructs from `('href "/foo") ('cls "p-5")` pairs
2. A tweak to `scm_apply` (or `Procedure.__call__`) to check if the last arg is a `KwArgs` instance, and if so, unpack it as `**kwargs` instead of a positional arg

Would you like to try writing the `@` builtin first, or the `scm_apply` change first?

```json {.usage}
{"model": "claude-sonnet-5", "prompt_tokens": 56874, "completion_tokens": 1060, "total_tokens": 57934, "cached_tokens": 55306, "cache_creation_tokens": 1566, "cost": 0.0093261}
```


what would we need to do to make that work?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Two changes, building on what we found in `core.py`:

**1. A builtin for `@`** — collect its args (each a `(key val)` pair, i.e. a Scheme list) into a dict-like object. Since regular Python dicts already come back fine from lisp (each pair is a 2-element list), the builtin just needs to build one:

```python
core._builtin['@'] = lambda *pairs: KwArgs(dict(pairs))
```

where `KwArgs` is a subclass of `dict` (or any marker class) so we can distinguish "this last arg represents kwargs" from an ordinary dict argument later.

**2. Detect and unpack `KwArgs` in the calling path** — both `scm_apply` (line 50) and `Procedure.__call__` currently do a flat `fn(*args)`. They'd need something like:

```python
def scm_apply(fn, args):
    kw = args[-1].d if args and isinstance(args[-1], KwArgs) else {}
    args = args[:-1] if kw else args
    ...
    return fn(*args, **kw)
```

That's the shape of it. Do you want to start with the `KwArgs` class and `@` builtin, or the `scm_apply` change?

```json {.usage}
{"model": "claude-sonnet-5", "prompt_tokens": 57160, "completion_tokens": 686, "total_tokens": 57846, "cached_tokens": 56872, "cache_creation_tokens": 286, "cost": 0.009088500000000001}
```


ok updated types and scm_apply - does this look right?
(in ../00_core.ipynb)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me look at the current state of that file:
- ⏳ `py(code="view_file('../00_core.ipynb', start_line=1)")` ⏳
- ⏳ `py(code="from rgapi.skill import *\nrg(r'''KwArgs|…")` ⏳


```json {.tool}
{
  "id": "toolu_01GU5ZjtYHyhqbLmet2qTm2f",
  "name": "py",
  "args": {
    "code": "from rgapi.skill import *\nrg(r'''KwArgs|scm_apply|^@''', root='..', include='00_core.ipynb', context=3)"
  },
  "result": "00_core.ipynb-185-   },\n00_core.ipynb-186-   \"outputs\": [],\n00_core.ipynb-187-   \"source\": [\n00_core.ipynb:188:    \"def scm_apply(fn, args):\\n\",\n00_core.ipynb-189-    \"    kw = {}\\n\",\n00_core.ipynb:190:    \"    if len(args) > 0 and isinstance(args[-1], KwArgs): \\n\",\n00_core.ipynb-191-    \"        kw.update(args[-1])\\n\",\n00_core.ipynb-192-    \"        args = args[:-1]\\n\",\n00_core.ipynb-193-    \"    if isinstance(fn, Procedure): return fn._thunk(*args, **kw)\\n\",\n00_core.ipynb-241-    \"    fn = scm_eval_tco(hd, env, sfs)\\n\",\n00_core.ipynb-242-    \"    if isinstance(fn, Macro): return Thunk(fn.fn(*body), env)\\n\",\n00_core.ipynb-243-    \"\\n\",\n00_core.ipynb:244:    \"    return scm_apply(fn, [scm_eval_tco(o, env, sfs) for o in body])\"\n00_core.ipynb-245-   ]\n00_core.ipynb-246-  },\n00_core.ipynb-247-  {"
}
```

Let me see the rest of that `scm_apply` cell (it got cut off at line 193) and find the `KwArgs` class definition:
- ⏳ `py(code="from fastcore.tools import view_file\npri…")` ⏳

- ⏳ `py(code="rg(r'''class KwArgs|^KwArgs''', root='..…")` ⏳


```json {.tool}
{
  "id": "toolu_01AFVWV5LS3pHeT2RrfM1MXi",
  "name": "py",
  "args": {
    "code": "from fastcore.tools import view_file\nprint(view_file('../00_core.ipynb', start_line=175, end_line=196))"
  },
  "result": "175:    ]\n176:   },\n177:   {\n178:    \"cell_type\": \"code\",\n179:    \"execution_count\": null,\n180:    \"id\": \"48103d34\",\n181:    \"metadata\": {\n182:     \"nbdev\": {\n183:      \"export\": \"true\"\n184:     }\n185:    },\n186:    \"outputs\": [],\n187:    \"source\": [\n188:     \"def scm_apply(fn, args):\\n\",\n189:     \"    kw = {}\\n\",\n190:     \"    if len(args) > 0 and isinstance(args[-1], KwArgs): \\n\",\n191:     \"        kw.update(args[-1])\\n\",\n192:     \"        args = args[:-1]\\n\",\n193:     \"    if isinstance(fn, Procedure): return fn._thunk(*args, **kw)\\n\",\n194:     \"    return fn(*args, **kw)\"\n195:    ]\n196:   },"
}
```



```json {.tool}
{
  "id": "toolu_012Zqc1PYczAa8cJG7knnxLv",
  "name": "py",
  "args": {
    "code": "rg(r'''class KwArgs|^KwArgs''', root='..', include='00_core.ipynb', context=5)"
  },
  "result": "No output"
}
```

The `scm_apply` code itself looks right — it correctly pops a trailing `KwArgs` instance off `args`, merges it into `kw`, and passes `**kw` to both the `Procedure._thunk` path and the plain-callable path. That matches what we discussed.

However, my search for a `KwArgs` class definition in `00_core.ipynb` came back empty — it isn't defined anywhere in that file. So `scm_apply` will raise a `NameError` as soon as it's called, since `KwArgs` doesn't exist yet.

Want me to search more broadly (other files in the repo) in case it's defined elsewhere, or do you still need to add the class itself?

```json {.usage}
{"model": "claude-sonnet-5", "prompt_tokens": 174796, "completion_tokens": 795, "total_tokens": 175591, "cached_tokens": 172974, "cache_creation_tokens": 1816, "cost": 0.02681565}
```


kwargs is defined in 01_types


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me check that file:
- ⏳ `py(code="rg(r'''class KwArgs|KwArgs''', root='..'…")` ⏳


```json {.tool}
{
  "id": "toolu_01T7jryu4rzx6N5DW9aJHYCA",
  "name": "py",
  "args": {
    "code": "rg(r'''class KwArgs|KwArgs''', root='..', include='01_types.ipynb', context=5)"
  },
  "result": "01_types.ipynb-99-     \"export\": \"true\"\n01_types.ipynb-100-    }\n01_types.ipynb-101-   },\n01_types.ipynb-102-   \"outputs\": [],\n01_types.ipynb-103-   \"source\": [\n01_types.ipynb:104:    \"class KwArgs(dict): pass\"\n01_types.ipynb-105-   ]\n01_types.ipynb-106-  },\n01_types.ipynb-107-  {\n01_types.ipynb-108-   \"cell_type\": \"code\",\n01_types.ipynb-109-   \"execution_count\": null,"
}
```

Found it — `class KwArgs(dict): pass`, defined in `01_types.ipynb` line 104, exported. That resolves the missing piece: it's a plain marker subclass of `dict`, and `scm_apply`'s `isinstance(args[-1], KwArgs)` check and `kw.update(args[-1])` will work correctly against it since it behaves exactly like a dict.

So together, `scm_apply` and `KwArgs` look right:
- `KwArgs(dict)` marks a dict as "these are kwargs" rather than a positional dict argument
- `scm_apply` pops a trailing `KwArgs` off `args`, merges it into `kw`, and passes `**kw` on both the `Procedure` and plain-callable branches

One thing to check: does the `@` builtin (that builds the `KwArgs` from alist-style pairs) exist yet? I don't see it in the `00_core.ipynb` search results so far.

```json {.usage}
{"model": "claude-sonnet-5", "prompt_tokens": 118854, "completion_tokens": 633, "total_tokens": 119487, "cached_tokens": 116427, "cache_creation_tokens": 2423, "cost": 0.018302850000000002}
```


In [ ]:
def kwargs(pairs): return {k.s if isinstance(k, Symbol) else k:v for k,v in pairs}

In [ ]:
def call(fn, args):
    pos = [a[0] for a in args if len(a) == 1]
    kw  = {k.s: v for k, v in (a for a in args if len(a) == 2)}
    return fn(*pos, **kw)

In [ ]:
"""
(call P `(("hello") (cls "p-5")))
""" @ lisp

<p class="p-5">hello</p>

In [ ]:
"""
(define (caar  p) (car (car p)))
(define (cadr  p) (car (cdr p)))
(define (cdar  p) (cdr (car p)))
(define (cddr  p) (cdr (cdr p)))
(define (caddr  p) (car (cdr (cdr p))))
(define (cadddr p) (car (cdr (cdr (cdr p)))))
(define (cddddr p) (cdr (cdr (cdr (cdr p)))))
""" @ lisp

'cddddr'

In [ ]:
"""
(define (make-post slug title date . body)
  (list 'post slug title date body))

(define (post? p)      (and (pair? p) (equal? (car p) 'post)))
(define (post-slug p)  (cadr p))
(define (post-title p) (caddr p))
(define (post-date p)  (cadddr p))
(define (post-body p)  (car (cddddr p)))
""" @ lisp

'post-body'

In [ ]:
"""
(define (fact n)
  (if (= n 0) 1
      (* n (fact (- n 1)))))

(make-post "recursion" "On Recursion" "2024-01-10"
  (list 'prose
    "Recursion is defining something in terms of itself. "
    "A procedure calls itself on a smaller input "
    "until it reaches a base case.")
  (list 'code-block
    "(define (fact n)"
    "  (if (= n 0) 1"
    "      (* n (fact (- n 1)))))")
  (list 'example "(fact 10)" (fact 10))
  (list 'footnote
    "This generates a recursive process — the chain of deferred "
    "multiplications grows linearly with n. An iterative version "
    "using an accumulator would not."))
""" @ lisp

[compact.types.Symbol(s='post'),
 'recursion',
 'On Recursion',
 '2024-01-10',
 [[compact.types.Symbol(s='prose'),
   'Recursion is defining something in terms of itself. ',
   'A procedure calls itself on a smaller input ',
   'until it reaches a base case.'],
  [compact.types.Symbol(s='code-block'),
   '(define (fact n)',
   '  (if (= n 0) 1',
   '      (* n (fact (- n 1)))))'],
  [compact.types.Symbol(s='example'), '(fact 10)', 3628800],
  [compact.types.Symbol(s='footnote'),
   'This generates a recursive process — the chain of deferred ',
   'multiplications grows linearly with n. An iterative version ',
   'using an accumulator would not.']]]

In [ ]:
"""
(define (my-length lst)
  (define (iter l acc)
    (if (null? l) acc
        (iter (cdr l) (+ acc 1))))
  (iter lst 0))

(make-post "lists" "On Lists" "2024-01-17"
  (list 'prose
    "A list is either empty, or a value consed onto another list. "
    "This definition is recursive, so many list operations are too.")
  (list 'code-block
    "(define (my-length lst)"
    "  (define (iter l acc)"
    "    (if (null? l) acc"
    "        (iter (cdr l) (+ acc 1))))"
    "  (iter lst 0))")
  (list 'example "(my-length '(a b c d e))" (my-length '(a b c d e)))
  (list 'footnote
    "my-length generates an iterative process despite being written "
    "recursively. The accumulator carries the state; "
    "the interpreter need not grow the stack."))
""" @ lisp

[compact.types.Symbol(s='post'),
 'lists',
 'On Lists',
 '2024-01-17',
 [[compact.types.Symbol(s='prose'),
   'A list is either empty, or a value consed onto another list. ',
   'This definition is recursive, so many list operations are too.'],
  [compact.types.Symbol(s='code-block'),
   '(define (my-length lst)',
   '  (define (iter l acc)',
   '    (if (null? l) acc',
   '        (iter (cdr l) (+ acc 1))))',
   '  (iter lst 0))'],
  [compact.types.Symbol(s='example'), "(my-length '(a b c d e))", 5],
  [compact.types.Symbol(s='footnote'),
   'my-length generates an iterative process despite being written ',
   'recursively. The accumulator carries the state; ',
   'the interpreter need not grow the stack.']]]

In [ ]:
"""
(define (render-node node)
  (let ((tag (car node))
        (content (cdr node)))
    (cond
      ((equal? tag 'prose)      (apply P content))
      ((equal? tag 'code-block) (Pre (apply Code content)))
      ((equal? tag 'example)    (Div (Code (car content))
                                     (Span (string-append " → " (number->string (cadr content))))))
      ((equal? tag 'footnote)   (apply Aside content))
      (else (error "unknown node type" tag)))))

(define (render-post p)
  (apply Main
    (H1 (post-title p))
    (P (post-date p))
    (map render-node (post-body p))))
""" @ lisp

'render-post'

In [ ]:
"""
(render-post (make-post "recursion" "On Recursion" "2024-01-10"
  (list 'prose "Recursion is defining something in terms of itself.")
  (list 'example "(fact 10)" (fact 10))))
""" @ lisp

<main><h1>On Recursion</h1><p>2024-01-10</p><p>Recursion is defining something in terms of itself.</p><div><code>(fact 10)</code><span> → 3628800</span></div></main>

In [ ]:
"""
(render-post (make-post "recursion" "On Recursion" "2024-01-10"
  (list 'prose
    "Recursion is defining something in terms of itself. "
    "A procedure calls itself on a smaller input "
    "until it reaches a base case.")
  (list 'code-block
    "(define (fact n)"
    "  (if (= n 0) 1"
    "      (* n (fact (- n 1)))))")
  (list 'example "(fact 10)" (fact 10))
  (list 'footnote
    "This generates a recursive process — the chain of deferred "
    "multiplications grows linearly with n. An iterative version "
    "using an accumulator would not.")))
""" @ lisp

<main><h1>On Recursion</h1><p>2024-01-10</p><p>Recursion is defining something in terms of itself. A procedure calls itself on a smaller input until it reaches a base case.</p><pre><code>(define (fact n)  (if (= n 0) 1      (* n (fact (- n 1)))))</code></pre><div><code>(fact 10)</code><span> → 3628800</span></div><aside>This generates a recursive process — the chain of deferred multiplications grows linearly with n. An iterative version using an accumulator would not.</aside></main>

In [ ]:
"""
(define *posts* '())

(define (register-post p)
  (set! *posts* (cons p *posts*)))

(define (find-post slug)
  (define (search ps)
    (cond ((null? ps) #f)
          ((equal? (post-slug (car ps)) slug) (car ps))
          (else (search (cdr ps)))))
  (search *posts*))

(define (all-posts) (reverse *posts*))
""" @ lisp

'all-posts'

In [ ]:
"""
(register-post (make-post "recursion" "On Recursion" "2024-01-10"
  (list 'prose
    "Recursion is defining something in terms of itself. "
    "A procedure calls itself on a smaller input "
    "until it reaches a base case.")
  (list 'code-block
    "(define (fact n)"
    "  (if (= n 0) 1"
    "      (* n (fact (- n 1)))))")
  (list 'example "(fact 10)" (fact 10))
  (list 'footnote
    "This generates a recursive process — the chain of deferred "
    "multiplications grows linearly with n. An iterative version "
    "using an accumulator would not.")))

(register-post (make-post "lists" "On Lists" "2024-01-17"
  (list 'prose
    "A list is either empty, or a value consed onto another list. "
    "This definition is recursive, so many list operations are too.")
  (list 'code-block
    "(define (my-length lst)"
    "  (define (iter l acc)"
    "    (if (null? l) acc"
    "        (iter (cdr l) (+ acc 1))))"
    "  (iter lst 0))")
  (list 'example "(my-length '(a b c d e))" (my-length '(a b c d e)))
  (list 'footnote
    "my-length generates an iterative process despite being written "
    "recursively. The accumulator carries the state; "
    "the interpreter need not grow the stack.")))
""" @ lisp

'*posts*'

In [ ]:
app, rt = fast_app()

@rt("/root")
def get():
    return Main(H1("A Lisp Blog"),
                Ul(*[Li(A(post_title(p), href=f"/root/post/{post_slug(p)}")) 
                     for p in all_posts()]))

@rt("/root/post/{slug}")
def get(slug: str):
    p = find_post(slug)
    if not p: return P("not found")
    return render_post(p)

serve()

In [ ]:
%%lisp
(call P `(("hello") (cls "p-5")))


<p class="p-5">hello</p>